# 📊 종목 단/중/장 + 기관급 분석 엔진 — Colab 퀵스타트

블랙록(BlackRock) Aladdin 류 기관급 분석 + 단/중/장 종목 분석을
한글 리포트로 출력합니다.

**진행 순서**
1. 패키지 업로드 & 압축 해제
2. 의존성 설치
3. 인터넷 없이 데모 실행 (합성 데이터)
4. 실제 종목 분석 (yfinance)
5. HTML 리포트 인라인 표시

> ⚠️ 모든 출력은 정보 제공 목적이며 투자 권유가 아닙니다.

## 1) engine_kr.zip 업로드 후 압축 해제
왼쪽 파일 탭에 `engine_kr.zip` 을 올린 뒤 아래 셀을 실행하세요.

In [ ]:
import zipfile, os

ZIP = 'engine_kr.zip'   # 업로드한 파일명
if os.path.exists(ZIP):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('.')
    print('압축 해제 완료')
else:
    print('engine_kr.zip 을 먼저 업로드하세요 (좌측 파일 탭)')

# 압축 안에 engine_kr/ 폴더가 생기면 그 안으로 이동
if os.path.isdir('engine_kr'):
    os.chdir('engine_kr')
print('작업 폴더:', os.getcwd())
print('파일:', sorted(os.listdir('.'))[:12])

## 2) 의존성 설치
(torch/xgboost 까지 모두 설치하므로 2~4분 소요될 수 있습니다)

In [ ]:
!pip install -q -r requirements.txt
print('설치 완료')

## 2-1) 한글 폰트 적용 (그래프 한글 깨짐 방지)
패키지에 한글 폰트가 **동봉**되어 있어 인터넷 없이도 적용됩니다.
한 번만 실행하면 이후 모든 차트의 한글이 정상 표시됩니다.

In [ ]:
from engine.report import setup_korean_font
fam = setup_korean_font(verbose=True)
print("적용된 한글 폰트:", fam)

# (선택) Colab 캐시 문제로 그래도 깨지면 아래 두 줄의 주석을 풀고 런타임 재시작
# !apt-get -qq install -y fonts-nanum > /dev/null && rm -rf ~/.cache/matplotlib
# print("나눔폰트 설치 완료 — 런타임 재시작 후 다시 실행하세요")

## 3) 인터넷 없이 데모 (합성 데이터)
엔진 전체 파이프라인이 도는지 먼저 확인합니다.
기관 스코어카드·몬테카를로·팩터·스트레스·부예측·리스크버짓이 모두 실행됩니다.

In [ ]:
from main import analyze

res = analyze('DEMO', use_synthetic=True, ml_model='rf')

sc = res['institutional']['scorecard']
print('\n기관 등급:', sc['overall_grade'], '| 점수:', sc['overall_score'])
print('의견:', sc['verdict'])
print('\n[몬테카를로 분석 글]')
print(res['institutional']['narratives']['montecarlo'])

## 4) 실제 종목 분석 (yfinance)
`ticker` 만 바꾸면 됩니다. 한국 종목은 `.KS`(코스피) / `.KQ`(코스닥).
- 미국: `AAPL`, `MSFT`, `NVDA`, `TSLA`
- 한국: `005930.KS`(삼성전자), `000660.KS`(SK하이닉스)
- 암호화폐: `BTC-USD`

In [ ]:
ticker = '005930.KS'   # ← 원하는 종목으로 변경

res = analyze(
    ticker,
    start='2018-01-01',
    ml_model='rf',          # rf | xgb | lstm | gru | transformer
    regime_method='kmeans', # kmeans | hmm | gmm
    initial_capital=10_000_000,  # 부 예측 원금 (1,000만원)
    target_vol=0.10,        # 리스크 버짓 목표 변동성
)

print('\n종합 시그널:', res['overall_signal'], res['overall_score'])
for k, v in res['institutional']['narratives'].items():
    if v:
        print(f'\n[{k}]\n{v}')

## 5) HTML 리포트를 셀에 바로 표시
기관 스코어카드(포트폴리오 카드), 모듈별 한글 분석 글,
단/중/장 미래 주가 몬테카를로 차트가 모두 포함된 리포트입니다.

In [ ]:
from IPython.display import HTML

html_path = res['report_paths']['html']
print('리포트 경로:', html_path)
HTML(open(html_path, encoding='utf-8').read())

---
### 참고
- 단기(60일)는 표본이 작아 ML/국면이 "데이터 부족"으로 생략될 수 있습니다(정상).
- 시장지수가 없으면 팩터 분해는 근사 프록시로 추정되며 리포트에 명시됩니다.
- 결과는 확률적 추정이며 미래 수익을 보장하지 않습니다.